# GenAI Workshop
## Lesson 2: GenAI Starter 

This lesson is intended to play around with prompting and model parameter settings. 

During this lesson you will learn how to ...

- use diverse roles for prompting
- apply different prompting patterns 
- manipulate the model completion via model parameters

### Set up the environment 

Before we can start, we have to setup the environment.  

In [ ]:
import os
from google import genai
from google.genai import types

# Check runtime environment to make sure we are running in a colab environment. 
if os.getenv("COLAB_RELEASE_TAG"):
   COLAB = True
   print("Running on COLAB environment.") 
else:
   COLAB = False
   print("WARNING: Running on LOCAL environment.")

In [ ]:
# Import colab specific lib to read user data (aka colab managed secrets)
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')  

In [ ]:
# Initialize Google GenAI Client API with GOOGLE_API_KEY to be able to call the model. 
client = genai.Client(api_key=GOOGLE_API_KEY)

In [ ]:
# Double check key settings by printing it out (or at least it length). 
if GOOGLE_API_KEY: 
    print(f' GOOGLE_API_KEY set with a length of {len(GOOGLE_API_KEY)}')
else: 
    print(f' ERROR: GOOGLE_API_KEY not set correctly!')

### Definition of convenient functions  

The three following methods will simplify to work with the GEMINI genai model.
For details see function documentation.   

In [ ]:
# set default values for model, model parameters and prompt
DEFAULT_MODEL = "gemini-2.5-flash-lite"
DEFAULT_CONFIG_TEMPERATURE = 0.9 
DEFAULT_CONFIG_TOP_K = 1
DEFAULT_CONFIG_TOP_P = 0.95
DEFAULT_CONFIG_MAX_OUTPUT_TOKENS = 500 
DEFAULT_SYSTEM_PROMPT = " "
DEFAULT_USER_PROMPT:str = " "

def generate_gemini_completion(
        model_name: str = DEFAULT_MODEL, 
        temperature: float = DEFAULT_CONFIG_TEMPERATURE,
        top_k: int = DEFAULT_CONFIG_TOP_K,
        top_p: float = DEFAULT_CONFIG_TOP_P, 
        max_output_tokens: int = DEFAULT_CONFIG_MAX_OUTPUT_TOKENS, 
        system_prompt : str  = DEFAULT_SYSTEM_PROMPT, 
        user_prompt : str = DEFAULT_USER_PROMPT,
        verbose: bool = False
        ) -> str: 
    
    """ Calls a gemini model with a given set of parameters and returns the completions 
    
    Parameters
    ----------
    model_name : str, optional [default: DEFAULT_GEMINI_MODEL]
        The name of the model to use for the completion
    temperature : float, optional [default: DEFAULT_CONFIG_TEMPERATURE]
        The temperature of the model
    top_k : int, optional [default: DEFAULT_CONFIG_TOP_K]
        The number of most recent matches to return
    top_p : float, optional [default: DEFAULT_CONFIG_TOP_P]
        The cumulative probability range of the best matches to return
    max_output_tokens : int, optional [default: DEFAULT_CONFIG_MAX_OUTPUT_TOKENS]
        The maximum number of output tokens to return
    system_prompt : str, optional [default: DEFAULT_SYSTEM_PROMPT]
        The system prompt to use for the completion
    user_prompt : str, optional [default: DEFAULT_USER_PROMPT]
        The user prompt to use for the completion
    verbose : bool, optional [default: False]
        Whether to print details of the completion process or not. Defaults to False            
    Returns 
    -------
    str :
        the generated text      
    """    
    if verbose: 
        # print out summary of input values / parameters
        print(f'Generating answer for following config:')
        print(f'  - SYSTEM PROMPT used:\n {system_prompt}')
        print(f'  - USER PROMPT used:\n {user_prompt}')
        print(f'  - MODEL used:\n {model_name} (temperature = {temperature}, top_k = {top_k}, max_output_tokens = {max_output_tokens})')

    # create generation config 
    model_config = types.GenerateContentConfig(
        max_output_tokens=max_output_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        system_instruction=system_prompt,
    )
    
    # create generation request
    response = client.models.generate_content(
        model=model_name,
        contents=user_prompt,
        config=model_config,
    )
    
    return response.text

### Exercise 01: Prompting with roles 

During this exercise you will how to use the different types of prompts.  

In [ ]:
user_prompt = "What is the most beautiful city in the world?"
system_prompt = "You are a friendly assistant with a preference for Germany."

In [ ]:
# TODO: Call genai model without any prompts
response = None
print(response)

In [ ]:
# TODO: Call genai model with user prompt only
response = None
print(response)

In [ ]:
# TODO: Call genai model with system and user prompt
response = None
print(response)

### Exercise 02: Prompting patterns and best practices

In this exercise, you will learn how to apply various prompting best practices to achieve the desired result. See [Prompt Engineering Guide](https://www.promptingguide.ai/techniques) for more information. 

#### Prompting parts 

To obtain the desired result from the genai model when prompting, try to take the following prompt proportions into account:  

- role: "As what kind of person should the model act?"
- context: "Are there any additional information that can help the model to answer my question?"
- question: "What is the task/action/question I ask for?" 
- output: "What kind of output (format) do I expect?"
- example(s): "Are there any helpful examples the model can use?"  

In [32]:
initial_prompt = "I want to go on holiday. Where should I go?"

In [ ]:
# Call genai model for completion with initial prompt.  
response = generate_gemini_completion(user_prompt=initial_prompt)
print(response)

In [ ]:
# TODO Create a better prompt following the 'prompting parts' best practices. 
prompt_role = ""
prompt_context = ""
prompt_question = ""
prompt_output = ""

In [ ]:
prompt_with_parts = f'{prompt_role} {prompt_context} {prompt_question} {prompt_output}'

In [ ]:
# TODO: Call genai model for completion with prompt with parts.  
response = None
print(response)

In [ ]:
# TODO 
#  Use system prompt to define the systems role in detail in addition 
#  to get an even better or more specific result. 
#  - personality: travel web site Wonder-World AI chatbot
#  - name: Wonder-World (always greet with your name ;-) )
#  - mission:  provide helpful queries for travelers.
#  - guardrails: 
#       - answer with advise only if question complies with mission 
#       - else say "Sorry I can't answer that question"  
system_prompt = ""

In [ ]:
# TODO: Call genai model for completion with prompt with parts and system prompt.  
response = None
print(response)

#### Chain of Thoughts 

Introduced in Wei et al. (2022), chain-of-thought (CoT) prompting enables complex reasoning capabilities through intermediate reasoning steps. You can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

In this exercise we want the genai model to determine if our statement is true or false.

In [ ]:
# This is the statement we want to check (as false) 
statement_to_evaluate = "The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1."

In [ ]:
# TODO 
#  Build up a chain of thoughts to help genai to create the right answer: 
#  
#  Use
#
#   - The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1. 
#  
#  and a corresponding explanation why this is false to help the genai model to answer 
#  the following statement correctly: 
#   
#   - The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.
chain_of_thought = ""
 
chain_of_thought_prompt = (
    chain_of_thought + 
    statement_to_evaluate + 
    "Answer:")

In [ ]:
# TODO: Call genai model for completion with chain of thought prompt as user prompt.  
response = None
print(response)

#### Few Shot Learning

While large-language models demonstrate remarkable zero-shot capabilities, they still fall short on more complex tasks when using the zero-shot setting. Few-shot prompting can be used as a technique to enable in-context learning where we provide demonstrations in the prompt to steer the model to better performance.

In this exercise we want the genai model to rate a given sentence as positive or negative.
Use few shot learning to support the genai model. 

In [ ]:
# This is the text to rate (as negative)
text_to_rate = "What a horrible show!" 

In [ ]:
# TODO
#   Build up a few shot prompt that helps the genai model to determine if a given  
#   statement is meant positive / negative.
few_shot_prompt = "" + text_to_rate

In [ ]:
# TODO: Call genai model for completion with few shot prompt as user prompt.
response = None
print(response)

### Exercise 03: Models and parameters

In this exercise, you will learn how to use the various genai model parameters to customise the result according to your wishes. 

In [ ]:
# You want to know why the color of the sky is blue.
model_parameter_prompt = "Why is the sky blue?"

In [ ]:
# TODO: Play around with different model parameter to get various answers  
temperature = 0.5
top_k = 10
top_p = 0.95
max_output_tokens = 200

In [ ]:
# Call genai model for completion with few shot prompt as user prompt.
response = generate_gemini_completion(user_prompt=model_parameter_prompt, temperature=temperature, top_k=top_k, top_p=top_p, max_output_tokens=max_output_tokens)
print(response)